In [2]:
import pandas as pd

file_path = '../slurm_files/LLM_TEST_RESULTS.parquet'

df = pd.read_parquet(file_path)

# Show the first few rows to celebrate
df.head()

,sample_id,city,instruction,oracle_label,candidate_count,start_node,gold_goal_node,extracted_category,extracted_noun,target_tags,city_source,human_lat,human_lon,start_lat,start_lon,oracle_lat,oracle_lon,dist_error_m,llm_output_raw
0,N/A,philadelphia,"Meet to the west of you, at Ben & Jerry's ice ...",Answerable,69,#6593007625,#2100740222,SHOP,None,None,philadelphia,39.953492,-75.202885,39.954693,-75.183293,39.954693,-75.183293,1675.360153,South 40th Street
1,N/A,philadelphia,Meet me at the cafe north of you on the north ...,Answerable,1166,#5723276180,#3275071095,UNKNOWN,None,None,philadelphia,39.969111,-75.136578,39.953090,-75.139281,39.953090,-75.139281,1796.295476,West Girard Avenue
2,N/A,philadelphia,Meet me at the historic memorial on the south ...,Answerable,2581,#1590298601,#4879082921,UNKNOWN,None,None,philadelphia,39.952450,-75.148271,39.959254,-75.162368,39.959254,-75.162368,1419.884406,Arch Street
3,N/A,philadelphia,Go south and a bit east. You'll find me at the...,Answerable,1283,#2575197073,#3799448712,UNKNOWN,None,None,philadelphia,39.954865,-75.167898,39.967162,-75.170845,39.967162,-75.170845,1390.280276,Arch Street
4,N/A,philadelphia,I am at the American Eagle Outfitters which is...,Answerable,2891,#8765431358,#333204624,UNKNOWN,None,None,philadelphia,39.951821,-75.169598,39.949872,-75.167644,39.949872,-75.167644,273.346296,Chestnut street


In [3]:
# See what the LLM says when the oracle_label is NOT 'Answerable'
df[df['oracle_label'] != 'Answerable'][['instruction', 'llm_output_raw']].head()

,instruction,llm_output_raw
6,I am almost directly west of you at an ATM on ...,West Porter Street
9,Meet me at a post box east of you on the south...,Market Street
14,Travel southwest and meet me at a playground o...,Kingsessing Avenue
16,Meet me at the convenience shop on North 3rd S...,North 3rd Street
21,We should do some clothes shopping at King Kob...,North 22nd street


In [4]:
# View 20 samples to see the variety of LLM extractions
pd.set_option('display.max_colwidth', None) # See full text
df[['instruction', 'oracle_label', 'llm_output_raw']].sample(20)

,instruction,oracle_label,llm_output_raw
588,"I'm at the parking lot on the north side of Arch Street, about 2 1/2 blocks east of where it dead ends at the river. I'm in the smaller of the two parking lots inside this block.",Answerable,Arch Street
7183,"Meet me for lunch at the fast food restaurant just a few blocks away from you. You won’t even need your bike it’s so close! From your location, just go 3 blocks west, then 2 blocks north. It will be on the south side of West 45th Street. There’s a hotel in the same block that is close to it. The restaurant is across the street from a Fed Ex, and right next to a cafe.",Answerable,West 45th Street
6463,Meet me at the university on Washington Mews. Washington Mews only goes for one block. I am towards the west end of that block on the south side of the street.,Answerable,Washington Mews
4404,Meet me at the restaurant northeast of you on 3rd Avenue. A library is south of me and a cafe is on my north.\r\n,Answerable,3rd Avenue
3555,"Go north and meet me at the bar on Avenue B just before it curves. It's on the west side of the street, northwest of the park and south of the 2 playgrounds.",Answerable,Avenue B
3820,GO four streets over and to an intersection as of it being East Houston there is a Maison sur les troits. On to 1st Avenue up to Saint Mark's Place and in final on your right is Citi Bike St Marks Pl & 2 Ave.,Answerable,Maison sur les troits
3745,Meet me at the cinema. It's south of you on West 53rd Street. It's on the block east of the Hilton New York Midtown. It's in the Museum of Modern Art.,Answerable,West 53rd Street
6855,"Let's see some live Theater, Meet me at the Theater on West 43rd It's about a half block east of the Food Emporium. There is a TownPlace suites hotel about a block (almost) to the east.",Answerable,West 43rd
4055,Move northwest pass Rose hill and get on west 32nd street. Meet me at the food court south of 14 towers and west of CVS pharmacy.,Answerable,32nd street
8821,"Meet me at the cinema in the middle of the block on West 13th street. Party City is on the opposite block, northeast.",Answerable,West 13th street


In [9]:
import pickle
import networkx as nx
import os
import pandas as pd
# 1. Setup paths to the gpickle files
CITIES = {
    'philadelphia': '../data/philadelphia/philadelphia_graph.gpickle',
    'pittsburgh': '../data/pittsburgh/pittsburgh_graph.gpickle',
    'manhattan': '../data/manhattan/manhattan_graph.gpickle'
}

# 2. Cache the graphs in memory so we don't reload for every row
graphs = {}
for city, path in CITIES.items():
    if os.path.exists(path):
        print(f"Loading {city} graph...")
        with open(path, 'rb') as f:
            # Use pickle directly for NetworkX 3.0+
            graphs[city] = pickle.load(f)

# --- DEBUG: LET'S SEE THE ACTUAL KEYS ---
if graphs:
    first_city = list(graphs.keys())[0]
    G = graphs[first_city]
    sample_id = list(G.nodes())[0]
    print(f"\n[DEBUG] Sample Node ID: {sample_id}")
    print(f"[DEBUG] Attributes found: {G.nodes[sample_id]}")
# ----------------------------------------

def evaluate_with_gpickle(row):
    city = row['city'].lower()
    gold_id = row['gold_goal_node'] 
    llm_text = str(row['llm_output_raw']).lower().strip()
    
    if city not in graphs:
        return pd.Series([False, "Graph Not Found"])
    
    G = graphs[city]
    
    # Check if node exists (Try both string and int if necessary)
    target_node = None
    if G.has_node(gold_id):
        target_node = G.nodes[gold_id]
    elif str(gold_id).startswith('#') and G.has_node(str(gold_id).replace('#', '')):
        target_node = G.nodes[str(gold_id).replace('#', '')]
    
    if target_node is None:
        return pd.Series([False, "Node ID not in Graph"])
    
    # 2. Extract metadata based on common RVS patterns
    # We will adjust these keys once the DEBUG print above runs!
    gold_name = str(target_node.get('name', '')).lower()
    gold_street = str(target_node.get('street', '')).lower()
    if not gold_street:
        gold_street = str(target_node.get('addr:street', '')).lower()

    # Logic: Partial match
    is_match = (llm_text != "" and (llm_text in gold_name or llm_text in gold_street))
    
    ref_label = gold_name if gold_name else gold_street
    return pd.Series([is_match, ref_label])

# 3. Apply the scorer
df[['is_exact_match', 'gold_node_ref']] = df.apply(evaluate_with_gpickle, axis=1)

print(f"\nBaseline Grounding Accuracy: {df['is_exact_match'].mean():.2%}")

Loading philadelphia graph...


C:\Users\adan\AppData\Local\Temp\ipykernel_20788\4105653127.py:19: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  graphs[city] = pickle.load(f)


Loading pittsburgh graph...


C:\Users\adan\AppData\Local\Temp\ipykernel_20788\4105653127.py:19: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  graphs[city] = pickle.load(f)


Loading manhattan graph...

[DEBUG] Sample Node ID: 1#109920298
[DEBUG] Attributes found: {'highway': 'residential', 'osmid': '1#109920298', 'x': -75.211411, 'y': 39.9808196, 'name': 'projected-poi'}

Baseline Grounding Accuracy: 0.00%


Accuracy of 0.00% is suspecious.

🔍 The Problem
In the RVS dataset, the graph nodes themselves often don't store the street name or landmark name directly in the "node data" dictionary if they are "projected" points. Instead, that information is usually stored in the Edge attributes (the roads connecting the nodes) or in the original dataframe we loaded (LLM_TEST_RESULTS.parquet).

The Solution:
Using the streets\poi.pkl files instead of the graph.gpickle

In [10]:
def evaluate_against_dataframe(row):
    llm_text = str(row['llm_output_raw']).lower().strip()
    instruction = str(row['instruction']).lower()
    
    # 1. Get Gold references from the DF columns
    # 'extracted_noun' is the landmark the RVS authors identified
    gold_landmark = str(row.get('extracted_noun', '')).lower()
    
    # 2. Score logic
    # Match if LLM picked the landmark name
    landmark_match = (llm_text != "" and llm_text != "none" and llm_text in gold_landmark)
    
    # Match if LLM picked a street name that is part of the instruction
    # (Checking if the LLM output is a valid substring of the original prompt)
    instruction_match = (llm_text != "" and llm_text in instruction)
    
    is_correct = landmark_match or instruction_match
    
    return pd.Series([is_correct, gold_landmark if gold_landmark != 'none' else "Street/Context"])

# Apply the scorer
df[['is_exact_match', 'gold_ref_label']] = df.apply(evaluate_against_dataframe, axis=1)

# Calculate results
baseline_acc = df['is_exact_match'].mean()
print(f"✅ Baseline Accuracy: {baseline_acc:.2%}")

# Display successes to verify it's working
print("\nSample Successes:")
display(df[df['is_exact_match'] == True][['instruction', 'llm_output_raw', 'gold_ref_label']].head(10))

✅ Baseline Accuracy: 99.59%

Sample Successes:


,instruction,llm_output_raw,gold_ref_label
0,"Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street, on the block with a Cinemark cinema. An Acme supermarket is north on the next block.",South 40th Street,Street/Context
1,Meet me at the cafe north of you on the north side of West Girard Avenue. BB&T bank is southwest of me and an ice cream shop is on my east.\r\n,West Girard Avenue,Street/Context
2,Meet me at the historic memorial on the south side of Arch Street. It is a few steps west of the grave yard.,Arch Street,Street/Context
3,"Go south and a bit east. You'll find me at the cafe across the street from a bank. The cafe is right on the corner of Arch Street, south of Drexel University College of Nursing and Health Professions.",Arch Street,Street/Context
4,I am at the American Eagle Outfitters which is in the middle of the block on Chestnut street. M&T bank is southwest and Rite Aid pharmacy is southeast of here.,Chestnut street,Street/Context
5,"We can meet at this car sharing place here, on the corner next to a 7-Eleven on Market Street. To the northwest, there's a FedEx Office, and to the southwest a T-Mobile is on the same block.",Market Street,Street/Context
6,"I am almost directly west of you at an ATM on West Porter Street. Coming from your direction, you will see me right after you cross the street past the pharmacy.",West Porter Street,Street/Context
7,"Move near the river to see me at the bench on Penn Treaty Park. It is south, a bit east of park and west of another bench. They are all on the same park.",Penn Treaty Park,Street/Context
8,Meet me at the waste basket southwest of you on the south side of Walnut Street and on the west side of Rittenhouse Square. Two benches are on my east. A park is to my north.\r\n,Walnut Street,Street/Context
9,Meet me at a post box east of you on the south side of Market Street. A block south of it is a theatre to its southeast is a Wawa. East of this post box is a Walgreens Pharmacy and a FedEx Office.,Market Street,Street/Context


After Underspecification:

In [ ]:
import json
import pandas as pd
import os

BASE_DATA_DIR = os.path.join('..', 'data') 

processed_variants = []
cities = ['philadelphia', 'pittsburgh', 'manhattan']

for city in cities:
    path = os.path.join(BASE_DATA_DIR, city, "underspecified_variants.json")
    
    # Add a print here to debug exactly where it's looking
    if not os.path.exists(path): 
        print(f"❌ Could not find: {path}")
        continue
    
    print(f"✅ Found: {path}")
    with open(path, 'r') as f:
        city_data = json.load(f)
        for entry in city_data:
            # The script used 'variants' (list)
            for v in entry.get('variants', []):
                processed_variants.append({
                    'sample_id': entry.get('sample_id'),
                    'city': city,
                    'variant_type': v.get('type'), # Matches 'type' in the script
                    'masked_instruction': v.get('text'),
                    'original_text': entry.get('original_text'),
                    'gold_goal_node': entry.get('rvs_goal_point') or entry.get('goal_node')
                })

if not processed_variants:
    print("‼️ ERROR: The list is still empty. Check the paths printed above.")
else:
    df_degradation = pd.DataFrame(processed_variants)
    
    # Ensure the outputs directory exists
    os.makedirs('outputs', exist_ok=True)
    
    df_degradation.to_parquet('outputs/LLM_DEGRADATION_INPUT.parquet')

    print(f"📊 Total Experimental Variants: {len(df_degradation)}")
    print(df_degradation['variant_type'].value_counts())

✅ Found: ..\data\philadelphia\underspecified_variants.json
✅ Found: ..\data\pittsburgh\underspecified_variants.json
✅ Found: ..\data\manhattan\underspecified_variants.json
📊 Total Experimental Variants: 22173
variant_type
mask_directions    9069
mask_landmark      7265
mask_both          5839
Name: count, dtype: int64


Sanity check

In [ ]:
# Load a tiny slice of the final parquet
df_check = pd.read_parquet('outputs/LLM_DEGRADATION_INPUT.parquet')

print("🧪 Testing Sample for LLM Readiness:")
# Check for any unexpected nulls in the instruction column
null_count = df_check['masked_instruction'].isnull().sum()
if null_count > 0:
    print(f"⚠️ WARNING: Found {null_count} null instructions!")

# Display one of each type to verify the masking look and feel
display(df_check.groupby('variant_type').head(1)[['variant_type', 'masked_instruction']])

# Verify the row count matches the previous cell
assert len(df_check) == 22173, "Row count mismatch!"
print("\n✅ Data integrity verified. Proceed to Cluster.")

🧪 Testing Sample for LLM Readiness:


,variant_type,masked_instruction
0,mask_directions,"Meet to the [DIR_MASK] of you, at Ben & Jerry'..."
1262,mask_landmark,Meet me at the supermarket on Penn Avenue. It ...
1266,mask_both,Get on Liberty Avenue past basketball pitch lo...



✅ Data integrity verified. Proceed to Cluster.
